# 02 — Validation des données et contrats

**Projet** : Résumé de comptes-rendus d'intervention — encodeur-décodeur appris, baseline extractive publiée

## Objectifs pédagogiques

1. Lire les contrats du projet : documents, résumés de référence, faits et table de prédictions publiée.
1. Comprendre pourquoi **trois** tables ne suffisent pas : ce qu'un contrat par table ne peut pas voir, et qui pourtant fausse un score.
1. Casser chaque règle volontairement pour lire le message d'échec qu'on lira en production.
1. Distinguer une violation de table (``SchemaError``) d'une incohérence croisée (``ValueError``) : les deux se corrigent différemment.

## 0. Environnement et corpus

Le notebook lit son propre corpus réduit (`outputs/notebooks/data`) ; s'il manque, il le génère et le persiste pour que les pipelines du notebook travaillent sur les mêmes fichiers que lui.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (200 comptes-rendus), un budget
# d'époques réduit et une architecture réduite : l'exécution reste de quelques secondes, et les
# valeurs absolues d'un notebook ne sont **pas** celles de la référence publiée par le README.
NB_DOCUMENTS = 200
NB_EPOCHS = 8

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "model.params.layers=1",
                "model.params.units=96",
                "model.params.pretraining.epochs=1",
                "+model.params.vocab_size=800",
                "+model.params.min_frequency=2",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
                "train.early_stopping.enabled=false",
                "seed=42",
                "log_level=WARNING",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Bac à sable : les notebooks génèrent **leur** corpus (réduit) et écrivent **leurs** artefacts sous
# `outputs/notebooks/`. `data/raw`, `artifacts/` et donc les chiffres publiés par le README restent
# ceux de `make data` / `make train` : aucun notebook ne peut les modifier par accident.
NB_ROOT = PROJECT_ROOT / "outputs" / "notebooks"
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=NB_ROOT / "data",
    artifacts_dir=NB_ROOT / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
ID_COLUMN = str(MODEL_NODE.get("id_column", "doc_id"))
COMPRESSION = float(MODEL_NODE.get("params", {}).get("compression", 0.45))
MAX_OUTPUT_TOKENS = int(MODEL_NODE.get("params", {}).get("max_output_tokens", 90))
STRATEGIES = list(MODEL_NODE.get("strategies_to_compare", []) or ["lead"])

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.45)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Stratégies comparées : {', '.join(STRATEGIES)}")
print(f"Budget de longueur: compression {COMPRESSION:.2f}, plafond {MAX_OUTPUT_TOKENS} tokens")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticSummaryCorpusGenerator
from src.data.loaders import SummaryCorpusLoader

LOADER = SummaryCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS, REFERENCES, FACTS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(
        f"Corpus du notebook lu : {len(DOCUMENTS)} comptes-rendus, "
        f"{len(REFERENCES)} résumés de référence, {len(FACTS)} faits annotés"
    )
else:
    # Un clone frais n'a pas encore de corpus réduit : le notebook le génère, de façon déterministe,
    # dans son bac à sable. `make data` produit le corpus de référence (600 comptes-rendus) dans
    # `data/raw` — les deux ne se mélangent jamais.
    bundle = SyntheticSummaryCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, REFERENCES, FACTS = bundle.documents, bundle.queries, bundle.facts
    METADATA = bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_references(REFERENCES)
    LOADER.save_facts(FACTS)
    LOADER.save_metadata(METADATA)
    print(f"Corpus réduit généré dans outputs/notebooks/data : {len(DOCUMENTS)} comptes-rendus")

TRAIN_DOCUMENTS, TRAIN_REFERENCES, TRAIN_FACTS = LOADER.load_split("train")
VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS = LOADER.load_split("val")
TEST_DOCUMENTS, TEST_REFERENCES, TEST_FACTS = LOADER.load_split("test")
SPLIT_SIZES = LOADER.split_sizes()
# Le test ne sert **jamais** à régler quoi que ce soit : les réglages se lisent sur la validation,
# le verdict sur le test, une seule fois.
GOLDEN = dict(zip(REFERENCES["doc_id"], REFERENCES["summary"], strict=False))
print("splits :", SPLIT_SIZES)
print("compression moyenne demandée :", round(float(REFERENCES["compression"].mean()), 4))

## 1. Les quatre contrats

Un contrat de données est du code exécutable : il nomme chaque colonne, son type et ses bornes. C'est la documentation la plus fiable du jeu de données, puisqu'elle est vérifiée à chaque exécution du pipeline.

In [ ]:
from src.data.schemas import (
    DocumentsSchema,
    FactsSchema,
    PredictedSummariesSchema,
    ReferenceSummariesSchema,
    validate_documents,
    validate_facts,
    validate_references,
)

for schema in (DocumentsSchema, ReferenceSummariesSchema, FactsSchema, PredictedSummariesSchema):
    print(f"{schema.__name__:26s} -> {', '.join(schema.to_schema().columns)}")

### 1.1 Le corpus respecte ses contrats

In [ ]:
validated_documents = validate_documents(DOCUMENTS)
validated_references = validate_references(REFERENCES)
validated_facts = validate_facts(FACTS)
print(
    f"{len(validated_documents)} documents, {len(validated_references)} résumés, "
    f"{len(validated_facts)} faits : les trois tables passent leurs contrats"
)

# Une propriété que le contrat par table **ne peut pas** vérifier : la surface d'un fait est bien
# l'extrait du document à la phrase annoncée. Elle est vérifiée ici, une fois, sur tout le corpus.
texts = dict(zip(validated_documents["doc_id"], validated_documents["text"], strict=False))
surfaces_ok = 0
for row in validated_facts.itertuples(index=False):
    if str(row.surface) in texts[str(row.doc_id)]:
        surfaces_ok += 1
print(f"surfaces retrouvées dans leur document : {surfaces_ok} / {len(validated_facts)}")
assert surfaces_ok == len(validated_facts), "une surface de fait n'est pas dans son document"

**Ce qu'il faut retenir**

- Le contrat des faits vérifie la **forme** (type connu, saillance binaire, index positif) ; la cellule ci-dessus vérifie la **substance** (la surface est bien dans le texte). Les deux sont nécessaires : un fait bien formé et faux reste faux.
- La table des prédictions est stricte sur ses colonnes : c'est le contrat publié, donc un ajout de colonne dans un rapport est un changement de contrat, pas un détail.
- Les résumés de référence sont validés sur leurs bornes (2 à 8 phrases) : un résumé d'une phrase serait un titre, pas un résumé, et fausserait la compression cible.

## 2. Les liens entre les tables

Ce qu'aucun contrat par table ne peut voir : un résumé de référence qui parle d'un document inconnu, un document que personne n'a résumé, un fait qui pointe au-delà des phrases de son document. Ces trois erreurs sont celles qui faussent un score sans jamais lever d'erreur — elles sont donc vérifiées par :func:`validate_corpus`.

In [ ]:
from src.data.schemas import validate_corpus

validate_corpus(DOCUMENTS, REFERENCES, FACTS)
print(
    "les trois tables sont cohérentes entre elles :",
    f"{len(DOCUMENTS)} documents, {len(REFERENCES)} références, {len(FACTS)} faits",
)

## 3. Casser volontairement les contrats

Chaque corruption ci-dessous correspond à un incident réel. Le message d'erreur est affiché tel quel — c'est lui qu'on lira en production, et c'est pour cela qu'il nomme le coupable.

In [ ]:
from typing import Any


def show_failure(label: str, call: Any) -> None:
    """Exécuter un validateur et afficher la première ligne de son erreur."""
    try:
        call()
    except Exception as error:  # on veut le message du contrat, quel qu'il soit
        print(f"[refusé] {label} : {str(error).splitlines()[0]}")
    else:
        print(f"[accepté] {label} : aucune erreur (le contrat est trop laxiste)")


# 1. un identifiant de document hors format
bad_id = DOCUMENTS.head(1).copy()
bad_id.loc[bad_id.index[0], "doc_id"] = "CR-1"
show_failure("identifiant hors format", lambda: validate_documents(bad_id))

# 2. une colonne manquante : la table n'est plus celle du contrat
show_failure("colonne manquante", lambda: validate_documents(DOCUMENTS.drop(columns=["text"])))

# 3. un type de fait inconnu
unknown_type = FACTS.head(1).copy()
unknown_type.loc[unknown_type.index[0], "fact_type"] = "humeur"
show_failure("type de fait inconnu", lambda: validate_facts(unknown_type))

# 4. une saillance qui n'est ni 0 ni 1
bad_salient = FACTS.head(1).copy()
bad_salient.loc[bad_salient.index[0], "salient"] = 2
show_failure("saillance hors {0, 1}", lambda: validate_facts(bad_salient))

In [ ]:
# 5. un résumé de référence orphelin : il parle d'un document qui n'existe pas
orphan = REFERENCES.copy()
orphan.loc[orphan.index[0], "doc_id"] = "CR-9999"
show_failure("résumé orphelin", lambda: validate_corpus(DOCUMENTS, orphan, FACTS))

# 6. un document que personne n'a résumé : il ne pourra jamais être mesuré
missing = REFERENCES.iloc[1:].copy()
show_failure("document sans référence", lambda: validate_corpus(DOCUMENTS, missing, FACTS))

# 7. un fait au-delà des phrases de son document
sizes = DOCUMENTS.set_index("doc_id")["n_sentences"]
target = str(sizes.idxmin())
outside = FACTS.copy()
outside.loc[outside["doc_id"] == target, "sentence_index"] = int(sizes.min())
show_failure(
    "fait hors des phrases de son document", lambda: validate_corpus(DOCUMENTS, REFERENCES, outside)
)

**Ce qu'il faut retenir**

- Les quatre premières corruptions lèvent un ``SchemaError`` (la table casse son propre contrat) ; les trois suivantes un ``ValueError`` (les tables se contredisent). Deux familles d'erreurs, deux corrections différentes.
- Un document sans référence est aussi grave qu'un résumé orphelin : dans les deux cas une jointure silencieuse ferait disparaître des lignes — et un score calculé sur moins de lignes n'est plus comparable.
- Un fait dont l'index dépasse les phrases du document est typique d'une annotation décalée après un retraitement du texte : le contrat le refuse au lieu de le rattacher à la dernière phrase.

## 4. Le contrat de la table publiée

Le pipeline d'inférence écrit une table de prédictions : identifiant, stratégie, résumé produit, longueur, ROUGE quand la référence est connue, couverture des faits et latence. C'est un **artefact publié** : son contrat est vérifié avant écriture (notebook 06).

In [ ]:
from src.inference.predictor import SummaryPredictor
from src.models.lead import LeadSummarizer

lead = LeadSummarizer(compression=COMPRESSION, max_output_tokens=MAX_OUTPUT_TOKENS, seed=42)
lead.fit(TRAIN_DOCUMENTS, TRAIN_REFERENCES)
pre_predictor = SummaryPredictor(
    lead,
    paths=NB_PATHS,
    config=CONFIG.model_dump(),
    text_column=TEXT_COLUMN,
    id_column=ID_COLUMN,
)
RAW_PREDICTIONS = pre_predictor.predict(TEST_DOCUMENTS)
# `save` écrit la table **publiée** (le contrat), puis on la relit comme le ferait un lecteur du
# rapport : ce qui est validé ici est exactement ce qui est sur le disque.
PUBLISHED = pd.read_csv(pre_predictor.save(RAW_PREDICTIONS), keep_default_na=False)
print("colonnes de la table publiée :", list(PUBLISHED.columns))
display(PUBLISHED.head(2).round(4))

In [ ]:
from src.data.schemas import validate_predictions

validated_predictions = validate_predictions(PUBLISHED)
print(f"{len(validated_predictions)} prédictions validées")

bad_prediction = PUBLISHED.copy()
bad_prediction.loc[bad_prediction.index[0], "doc_id"] = "pas-un-document"
show_failure("prédiction d'un document inconnu", lambda: validate_predictions(bad_prediction))

impossible_compression = PUBLISHED.copy()
impossible_compression.loc[impossible_compression.index[0], "compression"] = 1.4
show_failure(
    "compression supérieure à 1 (résumé plus long que le document)",
    lambda: validate_predictions(impossible_compression),
)

extra_column = PUBLISHED.copy()
extra_column["commentaire"] = "une colonne ajoutée à la main"
show_failure(
    "colonne non déclarée dans la table publiée", lambda: validate_predictions(extra_column)
)

**Ce qu'il faut retenir**

- La compression est bornée à 1,0 dans le contrat publié : un résumé plus long que son document n'est pas un résumé, et cette borne a déjà attrapé un décodeur qui recrachait ses propres jetons spéciaux.
- La table publiée est stricte : une colonne ajoutée par un analyste casse le contrat, donc elle est refusée **avant** d'être écrite — pas après, dans un dashboards qui lit des colonnes par position.
- Un résumé vide est autorisé par le contrat (c'est la baseline ``resume_vide``) : c'est la métrique qui doit être *définie* sur un système muet, pas le contrat qui doit l'interdire.